# 02 - Table 3 Baseline Comparison

Huấn luyện độc lập trên Colab từ artifacts của Notebook 01: Random Forest, XGBoost, Vanilla MLP và CyberDetect-MLP.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_PATH = '/content/drive/MyDrive/Nhom28_CyberDetect_MLP_Final'
os.makedirs(PROJECT_PATH, exist_ok=True)
%cd {PROJECT_PATH}
print('PROJECT_PATH =', PROJECT_PATH)

import time
from contextlib import contextmanager

@contextmanager
def step(name):
    t0 = time.time()
    print(f"\n[START] {name}", flush=True)
    try:
        yield
    finally:
        print(f"[DONE] {name} - {time.time() - t0:.1f}s", flush=True)


In [ ]:
!pip -q install pandas numpy scikit-learn tensorflow xgboost pyarrow matplotlib seaborn


In [ ]:

import os, json, time, math, random
import numpy as np, pandas as pd, tensorflow as tf
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dropout
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler
from tensorflow.keras.optimizers import Adam
try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None
OUT=f'{PROJECT_PATH}/data/colab_processed'; RESULTS=f'{PROJECT_PATH}/results'; MODELS=f'{PROJECT_PATH}/models'
os.makedirs(RESULTS, exist_ok=True); os.makedirs(MODELS, exist_ok=True)
with step('Load processed artifacts'):
    X_train=pd.read_parquet(f'{OUT}/X_train_top30.parquet').values.astype('float32'); X_test=pd.read_parquet(f'{OUT}/X_test_top30.parquet').values.astype('float32')
    y_train=pd.read_parquet(f'{OUT}/y_train.parquet')['label'].values.astype(int); y_test=pd.read_parquet(f'{OUT}/y_test.parquet')['label'].values.astype(int)
print(f'[INFO] X_train={X_train.shape}, X_test={X_test.shape}', flush=True)
meta=json.load(open(f'{MODELS}/colab_preprocessing_metadata.json', encoding='utf-8'))

def set_seed(seed=42): random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
def cosine(e, lr_min=1e-5, lr_max=1e-3, T=100): return lr_min + 0.5*(lr_max-lr_min)*(1+math.cos(math.pi*e/T))
def build_model(input_dim, variant='full'):
    model=Sequential([Input(shape=(input_dim,))])
    for u in [512,256,128]:
        model.add(Dense(u, activation='relu'))
        if variant not in ['vanilla','no_batchnorm']: model.add(BatchNormalization())
        if variant not in ['vanilla','no_dropout']: model.add(Dropout(0.3))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer=Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
    return model
def pred(model,X):
    p=model.predict(X,verbose=0).reshape(-1); return (p>=0.5).astype(int), p
def metrics(y, yp, prob): return {'accuracy':accuracy_score(y,yp),'precision':precision_score(y,yp,zero_division=0),'recall':recall_score(y,yp,zero_division=0),'f1':f1_score(y,yp,zero_division=0),'roc_auc':roc_auc_score(y,prob)}
rows=[]; preds={}
with step('Train Random Forest'):
    rf=RandomForestClassifier(n_estimators=200,max_depth=20,random_state=42,n_jobs=-1); t=time.time(); rf.fit(X_train,y_train); yp=rf.predict(X_test); prob=rf.predict_proba(X_test)[:,1]; m=metrics(y_test,yp,prob); m.update(model='Random Forest',train_time_sec=time.time()-t); rows.append(m); preds['Random Forest']=yp
if XGBClassifier:
    with step('Train XGBoost'):
        xgb=XGBClassifier(n_estimators=200,max_depth=8,learning_rate=0.1,objective='binary:logistic',eval_metric='logloss',random_state=42,n_jobs=-1); t=time.time(); xgb.fit(X_train,y_train); yp=xgb.predict(X_test); prob=xgb.predict_proba(X_test)[:,1]; m=metrics(y_test,yp,prob); m.update(model='XGBoost',train_time_sec=time.time()-t); rows.append(m); preds['XGBoost']=yp
for name,variant,cw in [('Vanilla MLP','vanilla',False),('CyberDetect-MLP','full',True)]:
    with step(f'Train {name}'):
        set_seed(42); model=build_model(X_train.shape[1],variant); callbacks=[EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)]
        if variant!='vanilla': callbacks.append(LearningRateScheduler(lambda e: cosine(e,T=100)))
        class_weight=None
        if cw:
            w=compute_class_weight('balanced',classes=np.unique(y_train),y=y_train); class_weight=dict(enumerate(w))
        t=time.time(); hist=model.fit(X_train,y_train,validation_split=0.2,epochs=100,batch_size=64,callbacks=callbacks,class_weight=class_weight,verbose=2)
        yp,prob=pred(model,X_test); m=metrics(y_test,yp,prob); m.update(model=name,train_time_sec=time.time()-t,epochs_ran=len(hist.history['loss'])); rows.append(m); preds[name]=yp
        if name=='CyberDetect-MLP': model.save(f'{MODELS}/colab_cyberdetect_mlp.h5')
res=pd.DataFrame(rows); res.to_csv(f'{RESULTS}/colab_table3_metrics.csv',index=False); display(res)
cm=confusion_matrix(y_test,preds['CyberDetect-MLP']); pd.DataFrame(cm).to_csv(f'{RESULTS}/colab_confusion_matrix.csv',index=False)
open(f'{RESULTS}/colab_classification_report.txt','w',encoding='utf-8').write(classification_report(y_test,preds['CyberDetect-MLP'],target_names=['Normal (0)','Attack (1)'],zero_division=0))
plot=res.set_index('model')[['accuracy','precision','recall','f1','roc_auc']]; ax=plot.plot(kind='bar',figsize=(11,5)); plt.ylim(0,1.05); plt.title('Table 3 Baseline Comparison'); plt.xticks(rotation=20,ha='right'); plt.tight_layout(); plt.savefig(f'{RESULTS}/fig_colab_table3_comparison.png',dpi=150); plt.show()
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=['Normal','Attack'],yticklabels=['Normal','Attack']); plt.title('CyberDetect-MLP Confusion Matrix'); plt.tight_layout(); plt.savefig(f'{RESULTS}/fig_colab_confusion_matrix.png',dpi=150); plt.show()
